In [17]:
import xgboost as xgb
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:

model = xgb.XGBClassifier()
model.load_model(r"D:\projects\creditcard_fraud\notebooks\models\xgboost_v1\model.json")

In [5]:
RAW_DATA_PATH = Path("data/creditcard_1.csv")
df = pd.read_csv(RAW_DATA_PATH)
df = df.sort_values("Time").reset_index(drop = True)

target_column = "Class"

split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

X_train = train_df.drop(columns=[target_column])
y_train = train_df[target_column]

X_test = test_df.drop(columns=[target_column])
y_test = test_df[target_column]

In [6]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

In [7]:
results = pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred,
    "fraud_probability": y_prob
})
results

,actual,predicted,fraud_probability
227845,0,0,1.158295e-05
227846,0,0,1.198055e-05
227847,0,0,1.205754e-08
227848,0,0,3.221699e-06
227849,0,0,4.404074e-04
...,...,...,...
284802,0,0,1.001791e-10
284803,0,0,5.324794e-06
284804,0,0,5.487861e-06
284805,0,0,5.505241e-07


In [9]:
fp_df = results[
    (results["actual"] == 0) &
    (results["predicted"] == 1)
].copy()

fn_df = results[
    (results["actual"] == 1) &
    (results["predicted"] == 0)
].copy()

In [10]:
print("False positives:", len(fp_df))
print("False negatives:", len(fn_df))

False positives: 12
False negatives: 19


In [11]:
print("Total test observations:", len(results))
print("False positives:", len(fp_df))
print("False negatives:", len(fn_df))
print("FP rate among legitimate:", len(fp_df) / (results["actual"] == 0).sum())
print("FN rate among fraud:", len(fn_df) / (results["actual"] == 1).sum())

Total test observations: 56962
False positives: 12
False negatives: 19
FP rate among legitimate: 0.00021094450401673494
FN rate among fraud: 0.25333333333333335


In [13]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

C:\Users\Abhishek Goswami\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
X_fp = X_test.loc[fp_df.index]
X_fn = X_test.loc[fn_df.index]

shap_fp = explainer(X_fp)
shap_fn = explainer(X_fn)

print("FP SHAP:", shap_fp.values.shape)
print("FN SHAP:", shap_fn.values.shape)

FP SHAP: (12, 30)
FN SHAP: (19, 30)


In [18]:
fp_shap_importance = pd.DataFrame({
    "feature": X_fp.columns,
    "mean_abs_shap": np.abs(shap_fp.values).mean(axis=0)
}).sort_values(
    "mean_abs_shap",
    ascending=False
)

fn_shap_importance = pd.DataFrame({
    "feature": X_fn.columns,
    "mean_abs_shap": np.abs(shap_fn.values).mean(axis=0)
}).sort_values(
    "mean_abs_shap",
    ascending=False
)

In [19]:
print("Top FP features:")
display(fp_shap_importance.head(15))

print("Top FN features:")
display(fn_shap_importance.head(15))

Top FP features:


,feature,mean_abs_shap
14,V14,4.178490
10,V10,1.415508
4,V4,1.220202
0,Time,0.974410
7,V7,0.927455
2,V2,0.832006
12,V12,0.777382
19,V19,0.630342
5,V5,0.577754
3,V3,0.517849


Top FN features:


,feature,mean_abs_shap
14,V14,1.795545
4,V4,1.273904
12,V12,1.140621
0,Time,1.074143
10,V10,0.998683
1,V1,0.972213
11,V11,0.595500
8,V8,0.542291
2,V2,0.465179
19,V19,0.434522


In [20]:
fp_signed = pd.DataFrame({
    "feature": X_fp.columns,
    "mean_shap": shap_fp.values.mean(axis=0),
    "mean_abs_shap": np.abs(shap_fp.values).mean(axis=0)
}).sort_values(
    "mean_abs_shap",
    ascending=False
)

fn_signed = pd.DataFrame({
    "feature": X_fn.columns,
    "mean_shap": shap_fn.values.mean(axis=0),
    "mean_abs_shap": np.abs(shap_fn.values).mean(axis=0)
}).sort_values(
    "mean_abs_shap",
    ascending=False
)

In [21]:
display(fp_signed.head(15))
display(fn_signed.head(15))

,feature,mean_shap,mean_abs_shap
14,V14,3.892420,4.178490
10,V10,0.911045,1.415508
4,V4,0.860058,1.220202
0,Time,-0.974410,0.974410
7,V7,-0.442884,0.927455
2,V2,-0.753193,0.832006
12,V12,0.354223,0.777382
19,V19,0.532382,0.630342
5,V5,-0.515439,0.577754
3,V3,0.517849,0.517849


,feature,mean_shap,mean_abs_shap
14,V14,0.659521,1.795545
4,V4,0.347885,1.273904
12,V12,-0.863575,1.140621
0,Time,-1.074143,1.074143
10,V10,-0.545138,0.998683
1,V1,-0.913357,0.972213
11,V11,-0.416968,0.595500
8,V8,-0.116446,0.542291
2,V2,-0.312256,0.465179
19,V19,0.123150,0.434522


In [22]:
fp_direction = pd.DataFrame(
    shap_fp.values,
    columns=X_fp.columns,
    index=X_fp.index
)

fp_toward_rate = (
    (fp_direction > 0)
    .mean()
    .sort_values(ascending=False)
)

In [23]:
fn_direction = pd.DataFrame(
    shap_fn.values,
    columns=X_fn.columns,
    index=X_fn.index
)

fn_toward_rate = (
    (fn_direction > 0)
    .mean()
    .sort_values(ascending=False)
)

In [24]:
display(fp_toward_rate.head(15))
display(fn_toward_rate.head(15))

V3        1.000000
V14       0.833333
V8        0.833333
V15       0.750000
V19       0.750000
V4        0.750000
V10       0.666667
V1        0.666667
V11       0.666667
Amount    0.666667
V12       0.500000
V26       0.500000
V25       0.500000
V24       0.500000
V17       0.416667
dtype: float64

V3        0.842105
V8        0.736842
V15       0.631579
V14       0.578947
V4        0.526316
V19       0.526316
V27       0.473684
V11       0.473684
Amount    0.473684
V2        0.421053
V26       0.421053
V22       0.421053
V5        0.368421
V6        0.368421
V21       0.368421
dtype: float64

In [25]:
fp_pattern_stats = pd.DataFrame({
    "mean_shap": shap_fp.values.mean(axis=0),
    "mean_abs_shap": np.abs(shap_fp.values).mean(axis=0),
    "toward_fraud_rate": (shap_fp.values > 0).mean(axis=0)
}, index=X_fp.columns)

fp_pattern_stats = fp_pattern_stats.sort_values(
    "mean_abs_shap",
    ascending=False
)

fn_pattern_stats = pd.DataFrame({
    "mean_shap": shap_fn.values.mean(axis=0),
    "mean_abs_shap": np.abs(shap_fn.values).mean(axis=0),
    "toward_fraud_rate": (shap_fn.values > 0).mean(axis=0)
}, index=X_fn.columns)

fn_pattern_stats = fn_pattern_stats.sort_values(
    "mean_abs_shap",
    ascending=False
)

In [27]:
print("FP Patern Stats")
display(fp_pattern_stats.head(15))
print("FN pattern Stats")
display(fn_pattern_stats.head(15))

FP Patern Stats


,mean_shap,mean_abs_shap,toward_fraud_rate
V14,3.892420,4.178490,0.833333
V10,0.911045,1.415508,0.666667
V4,0.860058,1.220202,0.750000
Time,-0.974410,0.974410,0.000000
V7,-0.442884,0.927455,0.250000
V2,-0.753193,0.832006,0.166667
V12,0.354223,0.777382,0.500000
V19,0.532382,0.630342,0.750000
V5,-0.515439,0.577754,0.083333
V3,0.517849,0.517849,1.000000


FN pattern Stats


,mean_shap,mean_abs_shap,toward_fraud_rate
V14,0.659521,1.795545,0.578947
V4,0.347885,1.273904,0.526316
V12,-0.863575,1.140621,0.210526
Time,-1.074143,1.074143,0.000000
V10,-0.545138,0.998683,0.157895
V1,-0.913357,0.972213,0.157895
V11,-0.416968,0.595500,0.473684
V8,-0.116446,0.542291,0.736842
V2,-0.312256,0.465179,0.421053
V19,0.123150,0.434522,0.526316


In [28]:
# Top FPs by V14 contribution
display(
    fp_direction[["V14", "V10", "V4", "Time"]]
    .join(fp_df["fraud_probability"])
    .sort_values("V14", ascending=False)
    .head(10)
)

,V14,V10,V4,Time,fraud_probability
274467,6.188727,1.791351,0.591288,-0.748909,0.999507
267760,5.977524,1.737038,1.453673,-1.245070,0.942966
266590,5.669188,1.630122,1.281388,-1.146591,0.970438
276063,5.540752,1.565137,1.455075,-1.294952,0.812879
274771,5.424939,1.943531,1.306288,-0.541865,0.955296
269400,5.389171,1.669275,-1.290210,-0.612229,0.869253
228505,5.359979,1.631742,1.277066,-1.132201,0.979516
256278,5.196463,-0.684509,1.853925,-1.214441,0.505164
274837,1.845070,-0.757674,1.661607,-1.109973,0.613857
262971,1.833647,-0.676503,1.601253,-1.246404,0.753948


In [29]:
display(
    fn_direction[["Time", "V1", "V10", "V12", "V14"]]
    .join(fn_df["fraud_probability"])
    .sort_values("Time")
    .head(10)
)

,Time,V1,V10,V12,V14,fraud_probability
231978,-1.554741,0.144399,-1.186778,-1.270061,2.202938,0.000256
251889,-1.520815,-0.948486,-0.643283,-0.553263,1.918780,0.081375
254395,-1.363745,-0.043644,-0.877942,0.226788,-1.430043,0.001204
229727,-1.316069,-2.680591,-0.791786,-0.326089,4.137103,0.042990
240222,-1.260907,-2.304059,-0.730735,-0.252755,3.680289,0.102733
281674,-1.251750,-2.679573,-0.689774,-1.898533,1.428103,0.000033
251880,-1.214494,-1.119256,-0.663274,-1.198400,1.774596,0.013865
238466,-1.198762,-0.656193,-0.779291,-0.939439,2.077593,0.002213
245347,-1.089379,-0.053196,-1.445996,-1.773307,-1.270829,0.000687
254344,-1.083440,-0.070227,0.409334,-2.028825,1.185185,0.002396
